# Bioinformatics Tools Walkthrough

> **A note before you start:** unlike every other notebook in this course,
> the code blocks below are **not runnable in this browser**. Every real
> tool in this notebook (QIIME2, DADA2, mothur, PICRUSt2) is a heavyweight,
> compiled, conda-based pipeline — the kind of software that needs its own
> Linux/Mac environment with gigabytes of dependencies. That's architecturally
> incompatible with a browser sandbox, no matter how the code is written.
> Read this notebook as a map of the real world, not an exercise.

## Where did all these CSVs actually come from?

Every dataset you've used so far — `example_gut_samples.csv`,
`real_gut_usa_malawi.csv` — is a finished **relative abundance table**: rows
of samples, columns of genera, numbers that sum to ~100%. Real microbiome
research starts much further back, from raw DNA sequencer output, and goes
through a pipeline to get there:

1. **Raw sequencing reads** (millions of short DNA fragments per sample, as FASTQ files)
2. **Quality control / trimming** (remove low-quality bases, primers)
3. **Denoising / OTU or ASV calling** (group near-identical sequences into "these came from the same organism")
4. **Taxonomy assignment** (match each ASV/OTU against a reference database to get a genus/species name)
5. **Abundance table** ← **this is the CSV you've been working with all along**
6. Downstream stats: diversity metrics (notebook 06), differential abundance testing (notebooks 03/04), visualization

QIIME2 is the tool most labs use to run steps 2–6 end to end today.

## QIIME2 — the dominant end-to-end platform

QIIME2 ("Quantitative Insights Into Microbial Ecology") wraps the whole
pipeline into a consistent command-line tool. Its two defining ideas:

- Every input/output is a typed **artifact** (`.qza` = "QIIME Zipped
  Artifact") that records its own provenance — you can always trace an
  output back to the exact commands and parameters that made it.
- Visualizations are separate artifacts (`.qzv`) you drag-and-drop into
  [view.qiime2.org](https://view.qiime2.org) to explore interactively —
  no local software needed just to LOOK at results.

A real (abbreviated) QIIME2 session looks like this:

```bash
# 1. Import raw FASTQ files into a QIIME2 artifact
qiime tools import \
  --type SampleData[PairedEndSequencesWithQuality] \
  --input-path manifest.csv --input-format PairedEndFastqManifestPhred33V2 \
  --output-path demux.qza

# 2. Denoise with DADA2 (see next section) — this single command
#    does trimming, error-correction, chimera removal, AND ASV calling
qiime dada2 denoise-paired \
  --i-demultiplexed-seqs demux.qza \
  --p-trim-left-f 13 --p-trunc-len-f 150 \
  --o-representative-sequences rep-seqs.qza \
  --o-table table.qza \
  --o-denoising-stats stats.qza

# 3. Assign taxonomy against a reference database (e.g. SILVA or Greengenes2)
qiime feature-classifier classify-sklearn \
  --i-classifier silva-138-nb-classifier.qza \
  --i-reads rep-seqs.qza \
  --o-classification taxonomy.qza

# 4. Compute diversity metrics — this is notebook 06, automated
qiime diversity core-metrics-phylogenetic \
  --i-phylogeny rooted-tree.qza --i-table table.qza \
  --p-sampling-depth 10000 --m-metadata-file sample-metadata.tsv \
  --output-dir core-metrics-results
```

Notice step 4: `core-metrics-phylogenetic` outputs Shannon diversity,
Faith's phylogenetic diversity, Bray-Curtis distances, and several other
metrics **automatically** — for every sample, in one command. Everything
notebook 06 built from scratch with numpy, real labs get from this one call
(plus phylogeny-aware variants that need an evolutionary tree, which is why
it's beyond a "by hand" teaching notebook).

## DADA2 — what "denoising" actually means

DADA2 is technically an R package, and it's also the algorithm QIIME2 calls
under the hood in the `qiime dada2 denoise-paired` step above. Understanding
what it does explains a term from notebook 01: **why ASVs replaced OTUs**.

- **The old approach (OTUs):** cluster sequences that are e.g. ≥97% similar
  into one bin, treat the bin as "one organism." Simple, but throws away
  real biological variation smaller than that 3% threshold, and the exact
  bins you get depend on which other samples happen to be in the batch.
- **DADA2's approach (ASVs):** model the sequencer's actual per-base error
  rates statistically, then ask "given this error model, which *exact*
  sequences are real biological variants, and which are just sequencing
  mistakes of a more abundant real sequence?" The output is exact sequences
  (ASVs), not similarity bins — reproducible across studies, and precise
  down to single-nucleotide differences.

This is also where chimeras (artificial hybrid sequences created during PCR
amplification) get detected and removed — another reason raw reads need a
whole pipeline before they're trustworthy numbers.

## mothur — the other major pipeline

**mothur** predates QIIME2 and is still used, especially in labs with
established mothur-based workflows. It covers the same conceptual ground
(QC → OTU/ASV calling → taxonomy → diversity) via its own command language
rather than QIIME2's plugin system. If you see a methods section citing
mothur instead of QIIME2, assume the same underlying biology and mostly
comparable results — the choice is largely lab convention/legacy, not a
sign of fundamentally different data.

## PICRUSt2 — predicting function from taxonomy

Everything so far tells you **who's there** (taxonomy, abundance). PICRUSt2
takes a further, riskier step: it tries to predict **what they're doing** —
which metabolic genes/pathways are likely present — by matching each ASV's
sequence to related organisms with *fully sequenced genomes* and inferring
likely gene content.

**The caveat matters more than the tool**: this is *inference*, not
measurement. Two organisms with near-identical 16S sequences can still have
meaningfully different gene content. Treat PICRUSt2 output (and any
"predicted function" claim in a microbiome paper) as a hypothesis-generating
step, not a direct measurement — exactly the kind of claim notebooks 03–04
trained you to interrogate rather than accept at face value.

### EXPLAIN #1

*In one or two sentences: what's the actual difference between what DADA2
measures and what PICRUSt2 predicts? Why does that difference matter when
you read a paper's conclusions?*

> your answer here

## Done — you now know the whole pipeline

Raw reads → QC → DADA2 denoising (ASVs) → taxonomy assignment → the
abundance-table CSVs you started this course with → the diversity metrics
you hand-built in notebook 06 → optionally, functional predictions via
PICRUSt2 (with real caveats). QIIME2 (or mothur) is the glue that runs most
of this end to end in real labs today.

**Next:** `08_gmwi2_wellness_index.ipynb` — a newer, single-number
health-status predictor built on top of this same kind of pipeline.
